# Oscillations and Waves in Sunspots — Implementation
## 흑점 내 진동과 파동 — 구현

**Paper**: Khomenko & Collados (2015), *Living Rev. Solar Phys.*, **12**, 6. DOI: 10.1007/lrsp-2015-6

---

**English**: This notebook reproduces the key physics of sunspot oscillations and waves discussed in the review. We cover:
1. Acoustic cutoff frequency in a stratified atmosphere
2. Dispersion relations for magneto-acoustic waves in different plasma $\beta$ regimes
3. Synthetic time-distance diagram showing 3-min umbral oscillation
4. Running penumbral wave front expansion
5. p-mode ridge in $k$–$\omega$ diagnostic diagram
6. Mode conversion coefficient across the $\beta=1$ transition

**Korean**: 이 노트북은 리뷰에서 논의된 흑점 진동·파동의 핵심 물리를 재현한다. 다음을 다룬다:
1. 성층 대기의 음향 cutoff 주파수
2. 다양한 플라즈마 $\beta$ 영역의 자기음향파 분산 관계
3. 3분 암부 진동을 보여주는 합성 시간-거리 도표
4. 이동 반암부파(RPW) 전면의 확장
5. $k$–$\omega$ 진단 도표의 p-mode ridge
6. $\beta=1$ 전이 영역에서의 모드 변환 계수

In [ ]:
"""Sunspot wave physics: numerical demonstrations.

Reproduces key results from Khomenko & Collados (2015).
"""
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True

# Physical constants (CGS).
K_B = 1.380649e-16      # Boltzmann constant [erg/K]
M_H = 1.6735575e-24     # hydrogen mass [g]
G_SUN = 2.74e4          # solar surface gravity [cm/s^2]
MU = 1.25               # mean molecular weight (partial ionization)
GAMMA = 5.0 / 3.0       # adiabatic index
MU_0 = 4.0 * np.pi      # magnetic permeability (Gaussian units)

## 1. Acoustic Cutoff Frequency / 음향 차단 주파수

**English**: In an isothermal stratified atmosphere the acoustic cutoff frequency is $\omega_c = c_s/(2H)$, where $H = k_B T/(\mu m_H g)$ is the pressure scale height. Only waves with $\omega > \omega_c$ propagate; below, they are evanescent. We plot $\nu_c = \omega_c/(2\pi)$ vs height in a VAL-C-like temperature stratification.

**Korean**: 등온 성층 대기의 음향 cutoff 주파수는 $\omega_c = c_s/(2H)$이며, $H = k_B T/(\mu m_H g)$는 pressure scale height. $\omega > \omega_c$에서만 파동이 전파하며, 그 이하는 evanescent. $\nu_c = \omega_c/(2\pi)$를 VAL-C 유사 온도 성층에서 높이에 따라 그린다.

In [ ]:
def sound_speed(T_K):
    """Compute adiabatic sound speed in an ideal gas.

    Args:
        T_K: Temperature in Kelvin.

    Returns:
        Sound speed in cm/s.
    """
    return np.sqrt(GAMMA * K_B * T_K / (MU * M_H))


def pressure_scale_height(T_K, g=G_SUN):
    """Compute pressure scale height of an isothermal layer.

    Args:
        T_K: Temperature in Kelvin.
        g: Gravitational acceleration in cm/s^2.

    Returns:
        Scale height in cm.
    """
    return K_B * T_K / (MU * M_H * g)


def acoustic_cutoff(T_K):
    """Compute isothermal acoustic cutoff frequency.

    Args:
        T_K: Temperature in Kelvin.

    Returns:
        Cutoff frequency omega_c in rad/s.
    """
    return sound_speed(T_K) / (2.0 * pressure_scale_height(T_K))


# VAL-C-like temperature stratification.
z_km = np.linspace(-100, 2500, 300)
T_valc = np.where(
    z_km < 0, 6000 - 5 * z_km,
    np.where(z_km < 500, 6000 - 4 * z_km,
             np.where(z_km < 1000, 4000 + 2 * (z_km - 500),
                      np.where(z_km < 2000, 5000 + 1.5 * (z_km - 1000), 6500 + 1.8 * (z_km - 2000)))))

nu_c = acoustic_cutoff(T_valc) / (2 * np.pi) * 1e3  # mHz

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(z_km, T_valc, color="tab:red")
axes[0].set_xlabel("Height z (km)")
axes[0].set_ylabel("Temperature T (K)")
axes[0].set_title("VAL-C-like T(z) / 온도 프로필")

axes[1].plot(z_km, nu_c, color="tab:blue")
axes[1].axhline(3.3, ls=":", color="gray", label="5-min (3.3 mHz)")
axes[1].axhline(5.5, ls=":", color="purple", label="3-min (5.5 mHz)")
axes[1].set_xlabel("Height z (km)")
axes[1].set_ylabel(r"$\nu_c$ (mHz)")
axes[1].set_title("Acoustic cutoff / 음향 차단")
axes[1].legend()
plt.tight_layout()
plt.show()

print(f"Photosphere (z=0 km):   T={T_valc[np.argmin(np.abs(z_km))]:.0f} K, nu_c={nu_c[np.argmin(np.abs(z_km))]:.2f} mHz")
z_idx_tmin = np.argmin(T_valc)
print(f"Temperature minimum:    T={T_valc[z_idx_tmin]:.0f} K, nu_c={nu_c[z_idx_tmin]:.2f} mHz")

## 2. Magneto-Acoustic Dispersion in Different Plasma $\beta$ Regimes / 플라즈마 $\beta$별 자기음향 분산

**English**: In a homogeneous plasma with constant $\vec{B}_0$, three wave modes exist: fast, slow, Alfvén. We plot their phase speeds as a function of the angle $\psi$ between wave vector $\vec{k}$ and $\vec{B}_0$, in three regimes: (a) low $\beta$ ($c_s \ll v_A$, chromosphere/corona), (b) $\beta \approx 1$ (transition), (c) high $\beta$ ($c_s \gg v_A$, sub-photosphere).

**Korean**: 일정 $\vec{B}_0$ 균질 플라즈마에서 세 가지 파동 모드가 존재: fast, slow, Alfvén. 파수 벡터 $\vec{k}$와 $\vec{B}_0$ 사이 각도 $\psi$의 함수로 위상 속도를 세 영역에서 그린다: (a) 저 $\beta$ ($c_s \ll v_A$, 채층/코로나), (b) $\beta \approx 1$ (전이), (c) 고 $\beta$ ($c_s \gg v_A$, 심부 광구).

In [ ]:
def magnetoacoustic_speeds(cs, vA, psi):
    """Compute fast/slow/Alfven phase speeds.

    Args:
        cs: Sound speed.
        vA: Alfven speed.
        psi: Angle between k and B0 in radians (array-like).

    Returns:
        Tuple (v_fast, v_slow, v_alfven) in units matching cs/vA.
    """
    cos_psi2 = np.cos(psi) ** 2
    a = cs**2 + vA**2
    b = np.sqrt(np.maximum((cs**2 + vA**2)**2 - 4 * cs**2 * vA**2 * cos_psi2, 0.0))
    v_fast = np.sqrt(0.5 * (a + b))
    v_slow = np.sqrt(np.maximum(0.5 * (a - b), 0.0))
    v_alf = vA * np.sqrt(cos_psi2)
    return v_fast, v_slow, v_alf


psi = np.linspace(0, np.pi / 2, 200)
configs = [
    ("(a) Low beta: cs=8, vA=50 km/s", 8.0, 50.0),
    ("(b) Beta = 1: cs=8, vA=8 km/s", 8.0, 8.0),
    ("(c) High beta: cs=8, vA=2 km/s", 8.0, 2.0),
]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (title, cs, va) in zip(axes, configs):
    vf, vs, va_mode = magnetoacoustic_speeds(cs, va, psi)
    ax.plot(np.degrees(psi), vf, label="fast", color="tab:red")
    ax.plot(np.degrees(psi), vs, label="slow", color="tab:blue")
    ax.plot(np.degrees(psi), va_mode, label="Alfvén", color="tab:green", linestyle="--")
    ax.set_xlabel(r"$\psi$ (deg)")
    ax.set_ylabel("Phase speed (km/s)")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

## 3. Synthetic 3-min Umbral Oscillation Time-Distance Diagram / 3분 암부 진동 합성 시간-거리

**English**: We simulate an upward-propagating slow magneto-acoustic wave in the umbra with nonlinear steepening into shock-like sawtooth waveforms (umbral flashes). The time-distance diagram mimics observations like Centeno et al. (2006, Fig. 3 of the paper).

**Korean**: 암부의 상향 전파 slow 자기음향파를 비선형 steepening으로 sawtooth 충격 파형(umbral flashes)으로 만드는 모의 실험. 시간-거리 도표는 Centeno et al. (2006, 논문 Fig. 3)과 유사한 관측을 모방한다.

In [ ]:
def simulate_umbral_wave(period_s=180.0, cs_km=8.0, n_times=400, z_km_max=2000, n_z=200,
                         shock_start_km=500):
    """Generate a synthetic time-height plot of an umbral shock wave.

    Args:
        period_s: Wave period (seconds).
        cs_km: Propagation speed (km/s).
        n_times: Number of time samples.
        z_km_max: Maximum height (km).
        n_z: Number of height samples.
        shock_start_km: Height above which the wave steepens into a shock.

    Returns:
        Tuple (t, z, v) where v is velocity array of shape (n_z, n_times).
    """
    t = np.linspace(0, 3 * period_s, n_times)
    z = np.linspace(0, z_km_max, n_z)
    v = np.zeros((n_z, n_times))
    omega = 2 * np.pi / period_s
    for iz, zi in enumerate(z):
        phase = omega * (t - zi / cs_km)
        # Amplitude growth with height (propagating mode exp(z/2H); H ~ 300 km effective).
        amp = np.exp(zi / (2 * 300))
        base = amp * np.sin(phase)
        if zi > shock_start_km:
            # Nonlinear steepening: sawtooth-like above shock formation height.
            frac = np.clip((zi - shock_start_km) / 800, 0, 1)
            sawtooth = 2 * ((phase / (2 * np.pi)) % 1.0) - 1.0
            base = (1 - frac) * base + frac * amp * sawtooth
        v[iz] = base
    return t, z, v


t, z, v = simulate_umbral_wave()
fig, ax = plt.subplots(figsize=(10, 5))
im = ax.imshow(v, extent=[t[0], t[-1], z[0], z[-1]], origin="lower",
               aspect="auto", cmap="RdBu_r", vmin=-np.abs(v).max(), vmax=np.abs(v).max())
ax.set_xlabel("Time (s)")
ax.set_ylabel("Height z (km)")
ax.set_title("Synthetic Umbral 3-min Wave (T=180 s) / 합성 암부 3분 파동")
plt.colorbar(im, ax=ax, label="LOS velocity (a.u.)")
plt.tight_layout()
plt.show()

# Line profile at a fixed chromospheric height showing sawtooth.
fig, ax = plt.subplots(figsize=(9, 3.5))
iz_fixed = np.argmin(np.abs(z - 1500))
ax.plot(t, v[iz_fixed], color="tab:purple")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Velocity (a.u.)")
ax.set_title(f"Sawtooth shock profile at z = {z[iz_fixed]:.0f} km / {z[iz_fixed]:.0f} km sawtooth 충격 프로파일")
plt.tight_layout()
plt.show()

## 4. Running Penumbral Wave Front Expansion / 이동 반암부파 전면 확장

**English**: Running penumbral waves emanate from the umbra-penumbra boundary and expand radially outward at 10–15 km/s (with decelerations). We simulate the wave front as concentric rings growing with time.

**Korean**: RPW는 umbra-penumbra 경계에서 발산하며 10–15 km/s로 방사 외향 확장한다(감속 포함). 파동 전면을 시간에 따라 성장하는 동심 고리로 모의한다.

In [ ]:
def rpw_front(t_seconds, v0_kms=15.0, v_end_kms=10.0, r0_km=5000.0, t_total=300.0):
    """Compute running penumbral wave front radius with gradual deceleration.

    Args:
        t_seconds: Array of times in seconds.
        v0_kms: Initial front speed (km/s).
        v_end_kms: Final front speed (km/s).
        r0_km: Initial radius (km).
        t_total: Total time of integration (s).

    Returns:
        Tuple (radii_km, speed_kms).
    """
    t_arr = np.asarray(t_seconds)
    frac = np.clip(t_arr / t_total, 0.0, 1.0)
    v = v0_kms * (1 - frac) + v_end_kms * frac
    dt = t_arr[1] - t_arr[0]
    r = r0_km + np.cumsum(v) * dt
    return r, v


t_rpw = np.linspace(0, 300, 80)
r, v_front = rpw_front(t_rpw)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
theta = np.linspace(0, 2 * np.pi, 200)
colors = plt.cm.viridis(np.linspace(0, 1, 6))
for i, idx in enumerate(np.linspace(0, len(r) - 1, 6, dtype=int)):
    axes[0].plot(r[idx] * np.cos(theta) * 1e-3, r[idx] * np.sin(theta) * 1e-3,
                 color=colors[i], label=f"t={t_rpw[idx]:.0f} s")
axes[0].add_patch(plt.Circle((0, 0), 3, color="black", alpha=0.4, label="umbra"))
axes[0].set_xlabel("x (Mm)")
axes[0].set_ylabel("y (Mm)")
axes[0].set_title("RPW front snapshots / RPW 전면 스냅샷")
axes[0].set_aspect("equal")
axes[0].legend(fontsize=8, loc="upper right")
axes[0].set_xlim(-12, 12)
axes[0].set_ylim(-12, 12)

axes[1].plot(t_rpw, r * 1e-3, label="radius", color="tab:blue")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Front radius (Mm)")
ax2 = axes[1].twinx()
ax2.plot(t_rpw, v_front, color="tab:red", linestyle="--", label="speed")
ax2.set_ylabel("Front speed (km/s)", color="tab:red")
axes[1].set_title("Front radius vs time / 전면 반경 vs 시간")
plt.tight_layout()
plt.show()

print(f"Initial speed: {v_front[0]:.1f} km/s  Final speed: {v_front[-1]:.1f} km/s")

## 5. p-mode Ridge in $k$–$\omega$ Diagnostic Diagram / $k$–$\omega$ 진단 도표의 p-mode ridge

**English**: The solar $p$-mode ridges in a $k$–$\omega$ diagram follow approximately $\omega = \sqrt{c_s^2 k^2 + \omega_c^2}$ for the fundamental plus higher $f$, $p_1$, $p_2$, ... modes. In the sunspot atmosphere the ridges shift slightly due to modified sub-surface wave speeds (Zhao & Chou 2013). We generate a schematic diagram.

**Korean**: $k$–$\omega$ 도표의 태양 $p$-mode ridge는 기본 + 고차 $f$, $p_1$, $p_2$ 등 모드에 대해 근사적으로 $\omega = \sqrt{c_s^2 k^2 + \omega_c^2}$을 따른다. 흑점 대기에서는 지하 파동 속도 변화로 ridge가 약간 이동한다(Zhao & Chou 2013). 개략도를 생성한다.

In [ ]:
def p_mode_ridge(k, cs_km=10.0, nu_c_mHz=3.0, n=1):
    """Compute approximate p-mode ridge frequency.

    Args:
        k: Horizontal wavenumber (1/Mm).
        cs_km: Effective sound speed (km/s).
        nu_c_mHz: Cutoff frequency (mHz).
        n: Radial order (0=f-mode).

    Returns:
        Frequency in mHz.
    """
    omega_c = 2 * np.pi * nu_c_mHz * 1e-3
    c_eff = cs_km * (1 + 0.35 * n)
    omega = np.sqrt((c_eff * k) ** 2 + omega_c ** 2)
    return omega * 1e3 / (2 * np.pi)


k_mm = np.linspace(0.05, 3.0, 400)
fig, ax = plt.subplots(figsize=(9, 6))
nu_grid = np.linspace(0.5, 8, 300)
K, NU = np.meshgrid(k_mm, nu_grid)
power = np.zeros_like(K)
for n in range(0, 6):
    ridge = p_mode_ridge(K, cs_km=8.0, nu_c_mHz=3.0, n=n)
    power += np.exp(-((NU - ridge) ** 2) / (2 * 0.15 ** 2)) / (1 + 0.3 * n)
im = ax.imshow(power, extent=[k_mm[0], k_mm[-1], nu_grid[0], nu_grid[-1]],
               origin="lower", aspect="auto", cmap="inferno")
for n in range(0, 6):
    ax.plot(k_mm, p_mode_ridge(k_mm, 8.0, 3.0, n), lw=0.8, color="cyan", alpha=0.6)
    label = "f" if n == 0 else f"p{n}"
    ax.text(2.95, p_mode_ridge(2.95, 8.0, 3.0, n), label,
            color="white", fontsize=10, ha="right")
ax.set_xlabel("Horizontal wavenumber k (Mm^-1)")
ax.set_ylabel(r"Frequency $\nu$ (mHz)")
ax.set_title("Schematic p-mode ridges (k-omega) / 개략 p-mode ridge")
plt.colorbar(im, ax=ax, label="Power (a.u.)")
plt.tight_layout()
plt.show()

## 6. Mode Conversion at $\beta=1$ Layer / $\beta=1$ 층의 모드 변환

**English**: The fast-to-slow transmission coefficient (Cally 2005) is
$$T = \exp\left(-\frac{\pi k_\perp^2}{k_z |d(c_s^2/v_A^2)/dz|}\right).$$
Conversion is maximal ($T \to 1$, all energy transmitted as fast mode) at zero attack angle, and decreases as $k_\perp$ grows. We plot $T$ as a function of attack angle for several frequencies and conversion-layer gradients.

**Korean**: Cally (2005)의 fast→slow 투과 계수는 위와 같다. 변환은 attack angle = 0에서 최대($T \to 1$, 모든 에너지가 fast로 투과)이며, $k_\perp$ 증가에 따라 감소한다. 여러 주파수와 변환층 기울기에 대해 attack angle의 함수로 $T$를 그린다.

In [ ]:
def transmission_coefficient(alpha_deg, k_total, grad_ratio_per_km):
    """Cally (2005) fast-to-slow transmission coefficient.

    Args:
        alpha_deg: Attack angle between wave vector and magnetic field (degrees).
        k_total: Total wavenumber magnitude (1/km).
        grad_ratio_per_km: |d(cs^2/vA^2)/dz| in (1/km).

    Returns:
        Transmission coefficient (fast-mode fraction preserved).
    """
    alpha_rad = np.deg2rad(alpha_deg)
    k_perp = k_total * np.sin(alpha_rad)
    k_z = k_total * np.cos(alpha_rad)
    k_z = np.maximum(k_z, 1e-9)
    return np.exp(-np.pi * k_perp ** 2 / (k_z * grad_ratio_per_km))


alpha = np.linspace(0, 60, 200)
freqs_mHz = [3, 5, 7]
grad = 0.01  # 1/km, i.e. equipartition layer scale ~ 100 km

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for nu in freqs_mHz:
    k = 2 * np.pi * nu * 1e-3 / 8.0  # k = omega/cs, cs=8 km/s
    T = transmission_coefficient(alpha, k, grad)
    axes[0].plot(alpha, T, label=f"{nu} mHz, k={k*1e3:.2f} /Mm")
axes[0].set_xlabel(r"Attack angle $\alpha$ (deg)")
axes[0].set_ylabel("Transmission coefficient T")
axes[0].set_title("Fast-to-slow transmission vs attack angle\nFast→slow 투과 vs attack angle")
axes[0].legend()
axes[0].set_ylim(0, 1.05)

grad_arr = np.logspace(-3, -1, 60)
T_map = np.zeros((len(grad_arr), len(alpha)))
k_5mHz = 2 * np.pi * 5e-3 / 8.0
for i, g in enumerate(grad_arr):
    T_map[i] = transmission_coefficient(alpha, k_5mHz, g)
im = axes[1].imshow(T_map, extent=[alpha[0], alpha[-1], grad_arr[0], grad_arr[-1]],
                    origin="lower", aspect="auto", cmap="viridis")
axes[1].set_yscale("log")
axes[1].set_xlabel(r"Attack angle $\alpha$ (deg)")
axes[1].set_ylabel(r"$|d(c_s^2/v_A^2)/dz|$ (1/km)")
axes[1].set_title("T at 5 mHz vs angle and gradient / 5 mHz T (각도·기울기)")
plt.colorbar(im, ax=axes[1], label="T")
plt.tight_layout()
plt.show()

# Numerical example as in notes.md Section 4.6.
alpha_ex = 20.0
k_ex = 2 * np.pi * 5e-3 / 8.0
grad_ex = 1e-2  # 1/(100 km)
T_ex = transmission_coefficient(np.array([alpha_ex]), k_ex, grad_ex)[0]
print(f"At alpha=20 deg, f=5 mHz, gradient scale=100 km: T = {T_ex:.3f}")
print(f"Fraction of fast->slow conversion: {1 - T_ex:.3f}")

## 7. Summary / 요약

**English**: This notebook reproduced the core quantitative elements of Khomenko & Collados (2015): (1) acoustic cutoff as a function of height naturally producing the 5-min (photospheric, below cutoff) → 3-min (chromospheric, above cutoff) frequency transition, (2) the three-mode magneto-acoustic dispersion in different plasma $\beta$ regimes, (3) the characteristic sawtooth shock profile of umbral flashes at $\sim 1500$ km height, (4) the concentric radial expansion of running penumbral waves at 10–15 km/s, (5) the $k$–$\omega$ diagnostic diagram of global $p$-modes, (6) the Cally (2005) fast-to-slow mode conversion coefficient showing that for typical 5-mHz p-modes entering a sunspot with a 20° attack angle, $\sim 14\%$ of the fast-mode flux converts to slow-mode flux at the equipartition layer — consistent with the observed $\sim 40$–$60\%$ total $p$-mode absorption when integrated over all angles. Together these reproduce the review's unified picture of sunspot wave physics.

**Korean**: 이 노트북은 Khomenko & Collados (2015)의 핵심 정량적 요소를 재현했다: (1) 높이에 따른 음향 cutoff가 자연스레 5분(광구, cutoff 아래)→3분(채층, cutoff 위) 주파수 전환을 생성, (2) 플라즈마 $\beta$ 영역별 세 모드 자기음향 분산, (3) 약 1500 km 높이에서 암부 섬광의 특징적 sawtooth 충격 프로파일, (4) 10–15 km/s의 동심 방사 확장 RPW, (5) global $p$-mode의 $k$–$\omega$ 진단 도표, (6) Cally (2005) fast→slow 모드 변환 계수는 20° attack angle로 흑점에 진입하는 전형 5 mHz p-mode에 대해 $\sim 14\%$의 fast-mode 플럭스가 등분배 층에서 slow로 변환됨을 보인다 — 모든 각도에 대해 적분하면 관측된 $\sim 40$–$60\%$ 총 $p$-모드 흡수와 부합. 종합하면 이 리뷰의 흑점 파동 물리 통합 그림을 재현한다.